# Qwen-3 0.6Bの教師あり微調整(SFT)をするサンプル

Qwen-3 0.6Bモデルを使用して、簡単な教師あり微調整(SFT)を行うサンプルコードです。

人間の解答例を用いてモデルを微調整することで、LLMがより自然な解答をできるようにする。SFTで利用されるデータセットは以下の様な形式の質問と回答の組み合わせです。

```json
{
  "conversations": [
    {"role": "user", "content": "質問文"},
    {"role": "assistant", "content": "解答文"}
  ]
}
```


In [ ]:
%pip install trl -q

In [ ]:
import torch
from tqdm.auto import tqdm
from datetime import datetime
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [ ]:
cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
use_bf16 = cap[0] >= 8
dtype = torch.bfloat16 if use_bf16 else torch.float16
bf16 = use_bf16
fp16 = not use_bf16

print(f"CUDA Device Capability: {cap}, Using bf16: {bf16}, fp16: {fp16}")

CUDA Device Capability: (7, 5), Using bf16: False, fp16: True


In [ ]:
model_id = "Qwen/Qwen3-0.6B"

print(f"Using model: {model_id}")

Using model: Qwen/Qwen3-0.6B


## データセットの準備

In [ ]:
# サンプルデータセットを読み込み
dataset = load_dataset("llm-jp/magpie-sft-v1.0", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [ ]:
# データセットの件数と学習データの割合を設定。サンプルでは10,000件だが時間がかかりすぎるので1,000件に制限
dataset_count = 1000

# content列だけを抽出してDataset化。
dataset = dataset.select(range(dataset_count))

# 学習に利用するために"conversations"カラム名を "messages" に統一してシャッフル
dataset = dataset.rename_column("conversations", "messages")

In [ ]:
# データの内容を確認
dataset[0]["messages"]

[{'role': 'user', 'content': 'カボチャを使ったスイーツのレシピをいくつか教えてください。'},
 {'role': 'assistant',
  'content': 'もちろんです、カボチャを使ったスイーツは秋にぴったりですね。以下にいくつかのレシピをご紹介します。\n\n1. カボチャのスフレパウンドケーキ\n- 材料：カボチャ 200g、生クリーム 50ml、牛乳 50ml、卵 3個、砂糖 100g、薄力粉 70g、バニラエッセンス 少々\n- 調理方法：カボチャを蒸してつぶし、生クリームと牛乳を加えて混ぜる。別のボウルで卵黄と砂糖を混ぜて、その中にカボチャミックスを加える。さらに薄力粉を加えて混ぜる。卵白を泡立て器で固まり始めたら、徐々に残りの砂糖を加えながらしっかりと泡立てる。カボチャミックスに卵白を加えて混ぜる。型に流し入れ、180度に予熱したオーブンで約40分焼く。\n\n2. カボチャのプディング\n- 材料：カボチャ 150g、牛乳 200ml、砂糖 大さじ4、ゼラチン 大さじ1、バニラエッセンス 少々\n- 調理方法：カボチャを蒸してつぶし、牛乳と砂糖を加えて混ぜる。ゼラチンを水に溶かし、その中にカボチャミックスを加える。バニラエッセンスを加えて混ぜ、型に流し入れて冷蔵庫で固める。\n\n3. カボチャのアイスクリーム\n- 材料：カボチャ 200g、牛乳 200ml、砂糖 大さじ4、バニラエッセンス 少々\n- 調理方法：カボチャを蒸してつぶし、牛乳、砂糖、バニラエッセンスを加えて混ぜる。フライパンで煮て、均一になるまで攪拌する。冷ましてから、アイスクリームメーカーで凍らせる。\n\nこれらのレシピをお試しください！それぞれ、季節感あふれる美味しいスイーツになりますよ。'}]

## 学習器の設定

In [ ]:
per_device_train_batch_size = 1 if fp16 else 2
gradient_accumulation_steps = 8  # 実効バッチを稼ぐ
warmup_ratio = 0.03
num_train_epochs = 10 if bf16 else 1  # 無料版は時間がかかるので1epoch

current_date = datetime.now().strftime("%Y%m%d")
output_dir_name = f"../results/continued-pretrain_{current_date}"

args = SFTConfig(
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    fp16=fp16,
    bf16=bf16,
    logging_steps=10,
    seed=42,
    output_dir=output_dir_name,
    report_to="tensorboard",
    save_total_limit=1,
    warmup_ratio=warmup_ratio,
)

In [ ]:
sft_trainer = SFTTrainer(
    model=model_id,
    args=args,
    train_dataset=dataset,
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [ ]:
sft_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


OutOfMemoryError: CUDA out of memory. Tried to allocate 406.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 156.12 MiB is free. Process 13924 has 8.42 GiB memory in use. Process 151409 has 6.17 GiB memory in use. Of the allocated memory 5.88 GiB is allocated by PyTorch, and 162.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {output_dir_name}